In [1]:
import kaggle_benchmarks as kbench
import pandas as pd
import csv, os, re, time


In [75]:
import shutil
import os
# Define source and destination
source_file = '/kaggle/input/datasets/muhammadasjadali/64hundo/emotion_captions (29).csv'
destination_file = '/kaggle/working/emotion_captions.csv'
# Copy a single file
shutil.copy(source_file, destination_file)

'/kaggle/working/emotion_captions.csv'

# Task and prompt builder

In [103]:
# @kbench.task(
#     name="Emotion Caption Generation — Flickr8k",
#     description="Generate 5 emotion-conditioned captions with guaranteed CSV saving."
# )
def generate_emotion_captions(llm) -> None:

    import pandas as pd
    import csv, os, re, time, json

    # ── Config ────────────────────────────────────────────────────────────────
    MOONDREAM_CSV = '/kaggle/input/datasets/muhammadasjadali/emotionconditionedcaptionsflickr8k/factual_captionsMD2 (5).csv'
    OUTPUT_CSV    = '/kaggle/working/emotion_captions.csv'
    EMOTIONS      = ['joyful', 'sad', 'tense', 'romantic', 'humorous']
    CSV_COLUMNS   = ['image_id', 'img_path', 'neutral_caption',
                     'emotion', 'emotion_id', 'emotion_caption', 'quality_score']
    KEY_MAP       = {'joyful': 'j', 'sad': 's', 'tense': 't', 'romantic': 'r', 'humorous': 'h'}
    MAX_RETRIES   = 3
    MIN_WORDS     = 15
    MAX_WORDS     = 30

    # ── Prompt ────────────────────────────────────────────────────────────────
    def build_prompt(neutral_caption, failed_emotions=None):
        """
        On retry, failed_emotions is a dict like:
          {'j': 'the too-short caption', 't': 'the too-long caption'}
        The prompt then tells the model exactly which ones to fix and why.
        """
        base = f"""You are an expert image captioner. Rewrite the description below in 5 emotional tones.

RULES — follow all strictly:

1. ANTI-HALLUCINATION:
   - Every subject, object, and action you write MUST be explicitly present in the original description.
   - You MAY omit details to meet the word limit, but MUST NOT add, invent, imply, or substitute anything.
   - Do not rename or generalize elements (e.g., do not replace "a wooden bench" with "a seat").

2. EMOTIONAL TONE (primary goal):
   - Each caption must feel distinctly and unmistakably like its assigned emotion.
   - Choose words, rhythm, and focus that make the emotion immediately felt by the reader.
   - Prioritize emotional impact in your word choices — cold vs. warm, heavy vs. light, sharp vs. soft.
   - The same scene should feel like a completely different experience across the 5 tones.

3. WORD COUNT:
   - Each caption must be between {MIN_WORDS} and {MAX_WORDS} words, inclusive.
   - Count your words before finalizing. If outside this range, rewrite until it fits.

4. LITERAL STYLE:
   - Plain, observational prose only. No poetry, no rhyme, no metaphor, no figurative language.
   - Convey emotion through concrete word choice and sentence structure — not hollow adverbs.
   - Do not use words like "joyfully", "sadly", or "tensely".

5. SENTENCE STRUCTURE:
   - Exactly one sentence per tone. Vary structure across tones — no repeated templates.
   - Sentence should be naturally structured.

6. OUTPUT FORMAT:
   - Return ONLY a valid JSON object with exactly 5 keys. No extra text outside the JSON.

Original Description: "{neutral_caption}"

TONE DEFINITIONS:
- "j" → joyful: warm, cheerful, vibrant
- "s" → sad: melancholic, heavy, desolate
- "t" → tense: urgent, sharp, suspenseful
- "r" → romantic: tender, affectionate, soft
- "h" → humorous: dry, witty, ironic, observational"""

        if failed_emotions:
            feedback_lines = "\n\nSOME CAPTIONS FROM YOUR LAST ATTEMPT VIOLATED THE WORD COUNT. FIX THEM:\n"
            for key, (caption, wc) in failed_emotions.items():
                direction = "too short" if wc < MIN_WORDS else "too long"
                feedback_lines += f'  - "{key}": was {wc} words ({direction}) → "{caption}"\n'
            feedback_lines += f"\nRewrite ALL 5 tones again. Every caption must be {MIN_WORDS}–{MAX_WORDS} words."
            base += feedback_lines

        base += """

{
  "j": "...",
  "s": "...",
  "t": "...",
  "r": "...",
  "h": "..."
}"""
        return base

    # ── Helpers ───────────────────────────────────────────────────────────────
    def word_count(text):
        return len(text.strip().split())

    def is_valid_length(text):
        return MIN_WORDS <= word_count(text) <= MAX_WORDS

    def compute_quality_score(caption, neutral_caption):
        score = 0.0
        if is_valid_length(caption):
            score += 0.5

        source_tokens  = set(re.findall(r'\b[a-z]{4,}\b', neutral_caption.lower()))
        caption_tokens = set(re.findall(r'\b[a-z]{4,}\b', caption.lower()))
        allowed_extras = {
            'that', 'with', 'from', 'this', 'their', 'they', 'each',
            'while', 'through', 'behind', 'though', 'where', 'there',
            'between', 'under', 'still', 'away', 'back', 'over', 'then',
            'look', 'seem', 'stand', 'walk', 'move', 'feel', 'watch',
            'slowly', 'quietly', 'barely', 'nearly', 'softly', 'gently'
        }
        real_new = (caption_tokens - source_tokens) - allowed_extras
        hallucination_penalty = min(len(real_new) * 0.1, 0.5)
        score += max(0.0, 0.5 - hallucination_penalty)
        return round(score, 3)

    def extract_captions(raw_text):
        text = raw_text.strip().removeprefix('```json').removesuffix('```').strip()
        try:
            data = json.loads(text)
            if isinstance(data, dict) and all(k in data for k in KEY_MAP.values()):
                return data
        except json.JSONDecodeError:
            pass
        extracted = {}
        for key in KEY_MAP.values():
            match = re.search(fr'"{key}"\s*:\s*"([^"]+)"', text)
            extracted[key] = match.group(1).strip() if match else ""
        return extracted

    def init_csv(path):
        if not os.path.exists(path):
            with open(path, 'w', newline='', encoding='utf-8') as f:
                csv.DictWriter(f, fieldnames=CSV_COLUMNS).writeheader()

    def append_rows(path, rows):
        with open(path, 'a', newline='', encoding='utf-8') as f:
            csv.DictWriter(f, fieldnames=CSV_COLUMNS).writerows(rows)

    # ── Load & clean data ─────────────────────────────────────────────────────
    df = pd.read_csv(MOONDREAM_CSV)

    for col in ['factual_caption', 'moondream_caption', 'caption', 'description']:
        if col in df.columns:
            df = df.rename(columns={col: 'factual_caption'})
            break

    if 'img_path' not in df.columns:
        df['img_path'] = df['image_id'].apply(
            lambda x: f'/kaggle/input/flickr8k/Images/{x}')

    df = df[
        df['factual_caption'].notna() &
        (df['factual_caption'].str.strip() != '')
    ].reset_index(drop=True)

    print(f"✅ Loaded {len(df)} images")

    # ── Resume logic ──────────────────────────────────────────────────────────
    init_csv(OUTPUT_CSV)
    if os.path.getsize(OUTPUT_CSV) > 100:
        done_ids = set(pd.read_csv(OUTPUT_CSV)['image_id'].unique())
        print(f"▶  Resuming — {len(done_ids)} images already done")
    else:
        done_ids = set()
        print("▶  Starting fresh")

    pending = df[~df['image_id'].isin(done_ids)].reset_index(drop=True)
    print(f"   Pending : {len(pending)} images ({len(pending) * 5:,} captions)")

    images_done        = 0
    session_start      = time.time()
    initial_done_count = len(done_ids)

    # ── Main loop ─────────────────────────────────────────────────────────────
    for _, row in pending.iterrows():
        image_id        = row['image_id']
        neutral_caption = str(row['factual_caption']).strip()
        img_path        = row['img_path']

        if len(neutral_caption.split()) < 3:
            continue

        print(f"\n{'─'*68}")
        print(f"  [{images_done + initial_done_count + 1}/{len(df)}]  {image_id}")
        print(f"  Neutral : {neutral_caption}")
        print(f"{'─'*68}")

        captions_dict  = {}
        failed_emotions = None  # populated on length violations

        # ── Retry loop (API errors + length violations) ───────────────────────
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                raw_response  = llm.prompt(build_prompt(neutral_caption, failed_emotions))
                captions_dict = extract_captions(raw_response)
            except Exception as e:
                print(f"      API error (attempt {attempt}/{MAX_RETRIES}): {e}")
                time.sleep(5)
                continue

            # Check which keys violate word count
            failed_emotions = {
                key: (cap, word_count(cap))
                for key, cap in captions_dict.items()
                if cap and not is_valid_length(cap)
            }

            if not failed_emotions:
                break  # All captions are valid — no retry needed

            print(f"      ⚠️  Attempt {attempt}: {len(failed_emotions)} caption(s) "
                  f"violated word count — retrying with feedback...")

        # ── Per-emotion saving ────────────────────────────────────────────────
        image_rows = []
        for emotion in EMOTIONS:
            raw_cap = captions_dict.get(KEY_MAP[emotion], "").strip()

            if not raw_cap or len(raw_cap) < 10:
                print(f"  ❌ [{emotion:10s}] → empty or too short, skipping")
                continue

            wc    = word_count(raw_cap)
            valid = is_valid_length(raw_cap)
            score = compute_quality_score(raw_cap, neutral_caption)

            status = "✅" if valid else "⚠️ "
            print(f"  {status} [{emotion:10s}] ({wc:2d}w, score={score}) → {raw_cap}")

            image_rows.append({
                'image_id':        image_id,
                'img_path':        img_path,
                'neutral_caption': neutral_caption,
                'emotion':         emotion,
                'emotion_id':      EMOTIONS.index(emotion),
                'emotion_caption': raw_cap,
                'quality_score':   score,
            })

        if image_rows:
            append_rows(OUTPUT_CSV, image_rows)
            done_ids.add(image_id)
            images_done += 1
            print(f"  💾  Written to CSV")

    # ── Final summary & assertions ────────────────────────────────────────────
    elapsed = time.time() - session_start
    df_out  = pd.read_csv(OUTPUT_CSV)

    print(f"\n{'═'*68}")
    print(f"✅  Complete — {df_out['image_id'].nunique()} images, "
          f"{len(df_out)} rows, {elapsed/3600:.2f}h")
    print(f"   Avg quality score : {df_out['quality_score'].mean():.3f}")
    print(f"   % in 15–30 words  : "
          f"{(df_out['emotion_caption'].str.split().str.len().between(15, 30)).mean()*100:.1f}%")

    kbench.assertions.assert_true(
        df_out['image_id'].nunique() >= len(df) * 0.90,
        expectation="Completed at least 90% of images."
    )
    kbench.assertions.assert_true(
        df_out['quality_score'].mean() >= 0.50,
        expectation="Mean quality score is meaningful (≥0.5)."
    )
    kbench.assertions.assert_true(
        (df_out['emotion_caption'].str.split().str.len().between(MIN_WORDS, MAX_WORDS)).mean() >= 0.80,
        expectation="At least 80% of captions are within the 15–30 word limit."
    )

In [29]:
!pip install matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 30.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 55.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 47.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [matplotlib]5 [matplotlib]


In [117]:
# ════════════════════════════════════════════════════════════════════════════
# Filter bad captions (multi-sentence is TOLERATED).
#
# Drops an image (all 5 of its emotion rows) if ANY of its captions has:
#   - word count outside [6, 35]
#   - a banned adverb (joyfully, sadly, tensely, ...)
#   - a metaphor / abstract-poetic pattern (tide of, dance of, ...)
#   - mean grounding (vocab overlap with neutral) below MIN_MEAN_GROUNDING
#
# Dropped images become "pending" again, so re-running your existing
# generate_emotion_captions cell will regenerate them.
# ════════════════════════════════════════════════════════════════════════════
import pandas as pd
import re
import shutil

# ── Config — tweak these ────────────────────────────────────────────────
CSV_PATH    = '/kaggle/working/emotion_captions.csv'
BACKUP_PATH = CSV_PATH.replace('.csv', '_BACKUP_before_filter.csv')

MIN_WORDS          = 6      # tolerate captions as short as 6 words
MAX_WORDS          = 45     # tolerate captions up to 35 words
MIN_MEAN_GROUNDING = 0.10   # per-image mean across the 5 emotion captions

# ── Load ────────────────────────────────────────────────────────────────
df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df):,} rows from {df['image_id'].nunique():,} images\n")

# ── Helpers ─────────────────────────────────────────────────────────────
STOP = set('''a an the is are was were be been being am i you he she it we they
this that these those of in on at to for with by from as and or but if while
has have had do does did will would can could should may might must shall'''.split())

def content_words(text):
    words = re.findall(r"[a-z']+", str(text).lower())
    return set(w for w in words if len(w) >= 3 and w not in STOP)

def grounding(neutral, emotion):
    n = content_words(neutral)
    e = content_words(emotion)
    return len(n & e) / len(n) if n else 0.0

BANNED_ADV = ['joyfully', 'sadly', 'tensely', 'romantically', 'humorously',
              'tenderly', 'cheerfully', 'mournfully', 'anxiously', 'lovingly']
BANNED_RE = re.compile(r'\b(' + '|'.join(BANNED_ADV) + r')\b', re.I)

# REFINED: Removed broad matches like 'amidst', 'weight of', 'beauty of', 
# and the possessive "\w+'s presence" regex which were causing false positives.
ABSTRACT_PATTERNS = [
    r'\btide of\b', r'\bdance of\b', r'\bsymphony of\b', r'\becho of\b',
    r'\bwhispers? of\b', r'\bcanvas of\b', r'\btapestry of\b', r'\bvoid of\b',
    r'\bcaress of\b', r'\bembrace of\b', r'\bsilent (warning|threat)\b',
    r'\bfleeting\b.*\bmoment\b', r'\bquiet resignation\b'
]
METAPHOR_RE = re.compile('|'.join(ABSTRACT_PATTERNS), re.I)

# ── Per-caption flags ───────────────────────────────────────────────────
df['_grounding']  = df.apply(
    lambda r: grounding(r['neutral_caption'], r['emotion_caption']), axis=1)
df['_word_count'] = df['emotion_caption'].astype(str).str.split().str.len()
df['_bad_len']    = ~df['_word_count'].between(MIN_WORDS, MAX_WORDS)
df['_banned']     = df['emotion_caption'].astype(str).apply(
    lambda x: bool(BANNED_RE.search(x)))
df['_metaphor']   = df['emotion_caption'].astype(str).apply(
    lambda x: bool(METAPHOR_RE.search(x)))

# ── Per-image aggregation (drop image if ANY caption fails) ─────────────
img = df.groupby('image_id').agg(
    mean_grounding = ('_grounding', 'mean'),
    has_bad_len    = ('_bad_len',   'any'),
    has_banned     = ('_banned',    'any'),
    has_metaphor   = ('_metaphor',  'any'),
).reset_index()

img['_low_ground'] = img['mean_grounding'] < MIN_MEAN_GROUNDING
img['_drop'] = (
    img['_low_ground'] | img['has_bad_len'] | img['has_banned'] | img['has_metaphor']
)

# ── Summary ─────────────────────────────────────────────────────────────
print(f"{'═'*70}")
print("DROP REASONS (image flagged if ANY of its 5 captions triggers)")
print('═'*70)
print(f"  word count outside [{MIN_WORDS},{MAX_WORDS}]      {img['has_bad_len'].sum():>8,}")
print(f"  banned adverb                       {img['has_banned'].sum():>8,}")
print(f"  metaphor pattern                    {img['has_metaphor'].sum():>8,}")
print(f"  mean grounding < {MIN_MEAN_GROUNDING:<18}  {img['_low_ground'].sum():>8,}")
print(f"  ─────────────────────────────────────────────────")
print(f"  TOTAL flagged for regen             {img['_drop'].sum():>8,}  "
      f"({img['_drop'].mean()*100:.1f}% of {len(img):,} images)")

# ── Show a few examples for each reason so you can sanity-check ─────────
def show(mask, label, n=4):
    flagged_imgs = set(img.loc[mask, 'image_id'])
    sample = df[df['image_id'].isin(flagged_imgs)].head(n)
    if len(sample):
        print(f"\n  Examples — {label}:")
        for _, r in sample.iterrows():
            print(f"    [{r['emotion']:9s}] ({r['_word_count']}w) "
                  f"{str(r['emotion_caption'])[:85]}")

show(img['has_bad_len'], 'word count outside range')
show(img['has_banned'],  'banned adverbs')
show(img['has_metaphor'],'metaphor patterns')
show(img['_low_ground'] & ~img['has_bad_len'] & ~img['has_banned'] & ~img['has_metaphor'],
     'low grounding only')

Loaded 35,250 rows from 6,919 images

══════════════════════════════════════════════════════════════════════
DROP REASONS (image flagged if ANY of its 5 captions triggers)
══════════════════════════════════════════════════════════════════════
  word count outside [6,45]           137
  banned adverb                            603
  metaphor pattern                         928
  mean grounding < 0.1                      347
  ─────────────────────────────────────────────────
  TOTAL flagged for regen                1,827  (26.4% of 6,919 images)

  Examples — word count outside range:
    [joyful   ] (23w) Beneath the shade of the tree, the woman in the purple hood joyfully savors a treat, 
    [sad      ] (29w) The woman, purple hood obscuring her face, sits alone on the bench, a small object he
    [tense    ] (28w) Hooded and concealed, the woman on the bench nervously puts something to her mouth, t
    [romantic ] (34w) Beneath the sheltering branches of the tree, the woman in the p

In [107]:

generate_emotion_captions.run(kbench.llms["google/gemini-2.0-flash"])

✅ Loaded 8091 images
▶  Resuming — 6891 images already done
   Pending : 1201 images (6,005 captions)

────────────────────────────────────────────────────────────────────
  [6892/8091]  3691592651_6e4e7f1da9.jpg
  Neutral : A girl in a red life jacket is climbing a white inflatable rock climbing wall. Another person is sitting on a blue inflatable mat near the wall. The wall is located near a body of water with trees and a distant building.
────────────────────────────────────────────────────────────────────
  ✅ [joyful    ] (20w, score=0.5) → A girl ascends the white wall in her red life jacket, sunshine sparkling on the water, a perfect, carefree climb.
  ✅ [sad       ] (21w, score=0.5) → The red life jacket, a stark point, and the unreachable top of the white wall remind me of a lonely climb.
  ✅ [tense     ] (19w, score=0.5) → She climbs, red against white, each move a desperate scramble as the water waits, the building watches, a threat.
  ✅ [romantic  ] (20w, score=0.5) → The gi

KeyboardInterrupt: 

--------------------------------------------------

--------------------------

In [ ]:
import pandas as pd

# Path to your generated CSV
CSV_PATH = '/kaggle/working/emotion_captions.csv'

# 1. Load the data
df = pd.read_csv(CSV_PATH)

# 2. Find duplicates based on the exact pair: image_id and emotion_id
# keep=False ensures it shows ALL the conflicting rows, not just the extras
duplicates = df[df.duplicated(subset=['image_id', 'emotion_id'], keep=False)]

if not duplicates.empty:
    print(f"⚠️ Found {len(duplicates)} overlapping rows!")
    
    # Sort them so the identical pairs are grouped right next to each other
    duplicates_sorted = duplicates.sort_values(by=['image_id', 'emotion_id'])
    
    # Using display() gives a nice readable table in Kaggle notebooks
    display(duplicates_sorted)
    
else:
    print("✅ No duplicates found! Every image_id + emotion_id pair is completely unique.")




# Filter and fix bad captions

In [114]:
# @kbench.task(
#     name="Filter & Regenerate Bad Captions — Flickr8k",
#     description="Identifies and fixes captions failing grounding, word count, or stylistic checks."
# )
def filter_and_regenerate_captions(llm) -> None:
    import pandas as pd
    import csv, os, re, time, json

    # ── Config ────────────────────────────────────────────────────────────────
    CSV_PATH    = '/kaggle/working/emotion_captions.csv'
    EMOTIONS    = ['joyful', 'sad', 'tense', 'romantic', 'humorous']
    CSV_COLUMNS = ['image_id', 'img_path', 'neutral_caption',
                   'emotion', 'emotion_id', 'emotion_caption', 'quality_score']
    KEY_MAP     = {'joyful': 'j', 'sad': 's', 'tense': 't', 'romantic': 'r', 'humorous': 'h'}
    MAX_RETRIES = 3
    
    # Generation & Filter Targets
    GEN_MIN_WORDS      = 15
    GEN_MAX_WORDS      = 30
    FILTER_MIN_WORDS   = 6
    FILTER_MAX_WORDS   = 45
    MIN_MEAN_GROUNDING = 0.10

    # ── Filter Helpers ────────────────────────────────────────────────────────
    STOP = set('''a an the is are was were be been being am i you he she it we they
    this that these those of in on at to for with by from as and or but if while
    has have had do does did will would can could should may might must shall'''.split())

    def content_words(text):
        words = re.findall(r"[a-z']+", str(text).lower())
        return set(w for w in words if len(w) >= 3 and w not in STOP)

    def grounding(neutral, emotion):
        n = content_words(neutral)
        e = content_words(emotion)
        return len(n & e) / len(n) if n else 0.0

    BANNED_ADV = ['joyfully', 'sadly', 'tensely', 'romantically', 'humorously',
                  'tenderly', 'cheerfully', 'mournfully', 'anxiously', 'lovingly']
    BANNED_RE = re.compile(r'\b(' + '|'.join(BANNED_ADV) + r')\b', re.I)

    ABSTRACT_PATTERNS = [
        r'\btide of\b', r'\bdance of\b', r'\bsymphony of\b', r'\becho of\b',
        r'\bwhispers? of\b', r'\bcanvas of\b', r'\btapestry of\b', r'\bvoid of\b',
        r'\bcaress of\b', r'\bembrace of\b', r'\bsilent (warning|threat)\b',
        r'\bfleeting\b.*\bmoment\b', r'\bquiet resignation\b'
    ]
    METAPHOR_RE = re.compile('|'.join(ABSTRACT_PATTERNS), re.I)

    # ── Prompt & Generation Helpers ───────────────────────────────────────────
    def build_prompt(neutral_caption, failed_emotions=None):
        base = f"""You are an expert image captioner. Rewrite the description below in 5 emotional tones.

RULES — follow all strictly:
1. ANTI-HALLUCINATION: Every subject/object MUST be in the original. No inventions.
2. EMOTIONAL TONE: Each caption must feel unmistakably like its assigned emotion.
3. WORD COUNT: Exactly {GEN_MIN_WORDS}–{GEN_MAX_WORDS} words per caption.
4. LITERAL STYLE: No metaphors, no poetry, no "hollow" adverbs (-ly words).
5. STRUCTURE: Exactly one sentence per tone.

Original Description: "{neutral_caption}"

TONE DEFINITIONS:
- "j" → joyful | "s" → sad | "t" → tense | "r" → romantic | "h" → humorous"""

        if failed_emotions:
            feedback_lines = "\n\nFIX THESE ERRORS FROM LAST ATTEMPT:\n"
            for key, (caption, wc) in failed_emotions.items():
                direction = "too short" if wc < GEN_MIN_WORDS else "too long"
                feedback_lines += f'  - "{key}": {wc} words ({direction}) → "{caption}"\n'
            base += feedback_lines

        base += '\n\nReturn ONLY JSON: {"j": "...", "s": "...", "t": "...", "r": "...", "h": "..."}'
        return base

    def word_count(text):
        return len(text.strip().split())

    def is_valid_length_gen(text):
        return GEN_MIN_WORDS <= word_count(text) <= GEN_MAX_WORDS

    def compute_quality_score(caption, neutral_caption):
        score = 0.5 if is_valid_length_gen(caption) else 0.0
        source_tokens  = set(re.findall(r'\b[a-z]{4,}\b', neutral_caption.lower()))
        caption_tokens = set(re.findall(r'\b[a-z]{4,}\b', caption.lower()))
        allowed_extras = {'that', 'with', 'from', 'this', 'their', 'they', 'while', 'through', 'look', 'seem', 'feel'}
        real_new = (caption_tokens - source_tokens) - allowed_extras
        score += max(0.0, 0.5 - min(len(real_new) * 0.1, 0.5))
        return round(score, 3)

    def extract_captions(raw_text):
        text = raw_text.strip().removeprefix('```json').removesuffix('```').strip()
        try:
            return json.loads(text)
        except:
            extracted = {}
            for key in KEY_MAP.values():
                match = re.search(fr'"{key}"\s*:\s*"([^"]+)"', text)
                extracted[key] = match.group(1).strip() if match else ""
            return extracted

    # ── 1. Load and Evaluate ──────────────────────────────────────────────────
    if not os.path.exists(CSV_PATH):
        print("❌ CSV not found at working directory.")
        return

    df = pd.read_csv(CSV_PATH)
    df['_grounding'] = df.apply(lambda r: grounding(r['neutral_caption'], r['emotion_caption']), axis=1)
    df['_word_count'] = df['emotion_caption'].astype(str).str.split().str.len()
    df['_bad_len']    = ~df['_word_count'].between(FILTER_MIN_WORDS, FILTER_MAX_WORDS)
    df['_banned']     = df['emotion_caption'].astype(str).apply(lambda x: bool(BANNED_RE.search(x)))
    df['_metaphor']   = df['emotion_caption'].astype(str).apply(lambda x: bool(METAPHOR_RE.search(x)))

    img_stats = df.groupby('image_id').agg(
        mean_grounding = ('_grounding', 'mean'),
        has_bad_len    = ('_bad_len', 'any'),
        has_banned     = ('_banned', 'any'),
        has_metaphor   = ('_metaphor', 'any'),
    ).reset_index()

    img_stats['_drop'] = (img_stats['mean_grounding'] < MIN_MEAN_GROUNDING) | \
                         img_stats['has_bad_len'] | img_stats['has_banned'] | img_stats['has_metaphor']

    bad_image_ids = list(img_stats[img_stats['_drop']]['image_id'])
    
    if not bad_image_ids:
        print("✅ No problematic captions found. CSV is clean!")
        return
        
    print(f"⚠️ Found {len(bad_image_ids)} images requiring regeneration.")

    # ── 2. Atomic Regeneration Loop ───────────────────────────────────────────
    for idx, image_id in enumerate(bad_image_ids):
        source_row = df[df['image_id'] == image_id].iloc[0]
        neutral_caption = str(source_row['neutral_caption']).strip()
        img_path        = source_row['img_path']

        print(f"\n[{idx + 1}/{len(bad_image_ids)}] Fixing: {image_id}")

        captions_dict = {}
        failed_emotions = None  

        for attempt in range(1, MAX_RETRIES + 1):
            try:
                raw_response  = llm.prompt(build_prompt(neutral_caption, failed_emotions))
                captions_dict = extract_captions(raw_response)
                
                failed_emotions = {k: (cap, word_count(cap)) 
                                   for k, cap in captions_dict.items() 
                                   if not is_valid_length_gen(cap)}
                
                if not failed_emotions: break
                print(f"  Attempt {attempt}: {len(failed_emotions)} length violations. Retrying...")
            except Exception as e:
                time.sleep(2)

        image_rows = []
        for emotion in EMOTIONS:
            raw_cap = captions_dict.get(KEY_MAP[emotion], "").strip()
            if not raw_cap: continue

            image_rows.append({
                'image_id': image_id, 'img_path': img_path,
                'neutral_caption': neutral_caption, 'emotion': emotion,
                'emotion_id': EMOTIONS.index(emotion), 'emotion_caption': raw_cap,
                'quality_score': compute_quality_score(raw_cap, neutral_caption)
            })

        if image_rows:
            # Atomic Swap: Read fresh, drop ID, concat new, save
            latest_df = pd.read_csv(CSV_PATH)
            latest_df = latest_df[latest_df['image_id'] != image_id]
            updated_df = pd.concat([latest_df, pd.DataFrame(image_rows)[CSV_COLUMNS]], ignore_index=True)
            updated_df.to_csv(CSV_PATH, index=False)
            print(f"  💾 Safely updated {image_id}")

    print("\n✅ Regeneration complete.")

# ── Execution ─────────────────────────────────────────────────────────────


In [115]:
filter_and_regenerate_captions.run(kbench.llms["google/gemini-2.0-flash"])

⚠️ Found 1827 images requiring regeneration.

[1/1827] Fixing: 1141739219_2c47195e4c.jpg


KeyboardInterrupt: 